In [81]:
import pandas as pd
import json
import re

#### Read in Exposure, Occ Details, LMI (wage + growth)

In [82]:
# # TEMP: Read in exposure file until full composite exposure in developed
# pca = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/TBL-Data-AI-Exposure-What-do-we-know-202602-Updated.xlsx", sheet_name="FA1", skiprows=6)
# pca = pca[["SOC2018.1",  "PCA Weighted Score.1",  "Z Score Variance.1"]]
# pca.head()

# # Requires: "title", "soc", "summary"
# onet_summ = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/ONET_30.2_Database/Occupation Data.xlsx")
# onet_summ["O*NET-SOC Code"] = onet_summ["O*NET-SOC Code"].apply(lambda x: re.sub(r'\.\d+$', '', x))

# # Requires: "medianWage", "employment" (may need to pull in Lightcast to backfill missing)
# wage = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/OEWS/Arizona 2024 OEWS.xlsx", skiprows=1, sheet_name="Annual")
# wage = wage[["SOC Code1", "50th Percentile Wage", "Rounded Employment"]]

# # Requires: "annualOpenings", "projectedGrowth", "typicalEducation"
# lmi = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/AI PEW 08182025.xlsx")

# lmi = lmi[["SOC Code", "Annual Total Openings", "Annual Percent Change", "EducationValue", "WorkExperienceValue", "GrowthRate"]]


from pathlib import Path

BASE = Path(r"C:\Users\301533\Documents\GitHub\AI\ai_impact_analysis\ai_skills")

# TEMP: Read in exposure file until full composite exposure is developed
pca = pd.read_excel(
    BASE / "TBL-Data-AI-Exposure-What-do-we-know-202602-Updated.xlsx",
    sheet_name="FA1",
    skiprows=6
)
pca = pca[["SOC2018.1", "PCA Weighted Score.1", "Z Score Variance.1"]]
pca.head()

# Requires: "title", "soc", "summary"
onet_summ = pd.read_excel(
    BASE / "ONET_30.2_Database" / "Occupation Data.xlsx"
)
onet_summ["O*NET-SOC Code"] = onet_summ["O*NET-SOC Code"].apply(
    lambda x: re.sub(r'\.\d+$', '', x)
)

# Requires: "medianWage", "employment"
wage = pd.read_excel(
    BASE / "OEWS" / "Arizona 2024 OEWS.xlsx",
    skiprows=1,
    sheet_name="Annual"
)
wage = wage[["SOC Code1", "50th Percentile Wage", "Rounded Employment"]]

# Requires: "annualOpenings", "projectedGrowth", "typicalEducation"
lmi = pd.read_excel(
    BASE / "AI PEW 08182025.xlsx"
)
lmi = lmi[[
    "SOC Code",
    "Annual Total Openings",
    "Annual Percent Change",
    "EducationValue",
    "WorkExperienceValue",
    "GrowthRate"
]]


In [83]:
lmi.head()

,SOC Code,Annual Total Openings,Annual Percent Change,EducationValue,WorkExperienceValue,GrowthRate
0,11-1011,462,0.014890,Bachelor's degree,5 years or more,1.4890
1,11-1021,8878,0.008106,Bachelor's degree,5 years or more,0.8106
2,11-1031,39,0.009885,Bachelor's degree,Less than 5 years,0.9885
3,11-2011,8,0.005038,Bachelor's degree,Less than 5 years,0.5038
4,11-2021,468,0.008244,Bachelor's degree,5 years or more,0.8244


#### Format columns for .json for readability

In [84]:
wage['50th Percentile Wage'] = pd.to_numeric(wage['50th Percentile Wage'], errors='coerce')
wage['50th Percentile Wage'] = wage['50th Percentile Wage'].map('${:,.0f}'.format)

wage['Rounded Employment'] = pd.to_numeric(wage['Rounded Employment'], errors='coerce').fillna(0).astype(int)
wage['Rounded Employment'] = wage['Rounded Employment'].map('{:,d}'.format)

lmi['Annual Total Openings'] = pd.to_numeric(lmi['Annual Total Openings'], errors='coerce').fillna(0).astype(int)
lmi['Annual Total Openings'] = lmi['Annual Total Openings'].map('{:,d}'.format)

lmi["Annual Percent Change"] = pd.to_numeric(lmi['Annual Percent Change'], errors='coerce').fillna(0).astype(float)*100
lmi["Annual Percent Change"] = lmi["Annual Percent Change"].round(2)
lmi['Annual Percent Change'] = lmi['Annual Percent Change'].apply(lambda x: str(x) + '%')

lmi.loc[lmi['EducationValue'] == "High school diplo", 'EducationValue'] = "High school diploma"
lmi.loc[lmi['EducationValue'] == "No formal educati", 'EducationValue'] = "No formal education"
lmi.loc[lmi['EducationValue'] == "Doctoral or profe", 'EducationValue'] = "Doctoral or professional degree"
lmi.loc[lmi['EducationValue'] == "Postsecondary non", 'EducationValue'] = "Postsecondary non-degree"
lmi.loc[lmi['EducationValue'] == "Associate's degre", 'EducationValue'] = "Associate's degree"
lmi.loc[lmi['EducationValue'] == "Some college, no", 'EducationValue'] = "Some college, no degree"


In [85]:
lmi.head()

,SOC Code,Annual Total Openings,Annual Percent Change,EducationValue,WorkExperienceValue,GrowthRate
0,11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890
1,11-1021,"8,878",0.81%,Bachelor's degree,5 years or more,0.8106
2,11-1031,39,0.99%,Bachelor's degree,Less than 5 years,0.9885
3,11-2011,8,0.5%,Bachelor's degree,Less than 5 years,0.5038
4,11-2021,468,0.82%,Bachelor's degree,5 years or more,0.8244


#### Merge files (Note that we repeat on the 8-digit)

In [86]:
df = pd.merge(pca, onet_summ, left_on="SOC2018.1", right_on="O*NET-SOC Code", how="inner")
df = pd.merge(df, lmi, left_on="SOC2018.1", right_on="SOC Code", how="inner")
df = pd.merge(df, wage, left_on="SOC2018.1", right_on="SOC Code1", how="left")

# Create "exposure" based on PCA Weighted Score.1 quintiles:
df["exposure"] = pd.qcut(df["PCA Weighted Score.1"], q=5, labels=["Very Low", "Low", "Medium", "High", "Very High"])


In [87]:
df.head()

,SOC2018.1,PCA Weighted Score.1,Z Score Variance.1,O*NET-SOC Code,Title,Description,SOC Code,Annual Total Openings,Annual Percent Change,EducationValue,WorkExperienceValue,GrowthRate,SOC Code1,50th Percentile Wage,Rounded Employment,exposure
0,11-1011,0.818306,0.395745,11-1011,Chief Executives,Determine and formulate policies and provide o...,11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High
1,11-1011,0.818306,0.395745,11-1011,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High
2,11-1021,0.696313,0.213237,11-1021,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",11-1021,"8,878",0.81%,Bachelor's degree,5 years or more,0.8106,11-1021,"$90,000","100,340",High
3,11-2011,0.678133,0.185190,11-2011,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",11-2011,8,0.5%,Bachelor's degree,Less than 5 years,0.5038,11-2011,"$107,000",50,High
4,11-2021,0.783175,0.237888,11-2021,Marketing Managers,"Plan, direct, or coordinate marketing policies...",11-2021,468,0.82%,Bachelor's degree,5 years or more,0.8244,11-2021,"$135,920","4,350",High


In [88]:
# # Requires "relatedOccupationIds"
# related = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/ONET_30.2_Database/Related Occupations.xlsx")


# Requires "relatedOccupationIds"
related = pd.read_excel(
    BASE / "ONET_30.2_Database" / "Related Occupations.xlsx"
)

# Filter out records not ending in "00" in SOC Code1
related = related[related["O*NET-SOC Code"].astype(str).str.endswith("00")]

# Drop last 3 characters from O*NET-SOC Code
related["O*NET-SOC Code"] = related["O*NET-SOC Code"].str[:-3]
related["Related O*NET-SOC Code"] = related["Related O*NET-SOC Code"].str[:-3]

# Merge in "PCA Weighted Score.1" from df and filter out "Very High" and "High" exposure occupations
related = pd.merge(related, df[["SOC2018.1", "exposure"]], left_on="O*NET-SOC Code", right_on="SOC2018.1", how="left")
related = related[related["exposure"].isin(["Very High", "High"])]

# Merge in "PCA Weighted Score.1" from df for related occupations
related = pd.merge(
    related,
    df[["SOC2018.1", "exposure"]].rename(columns={"exposure": "related_exposure", "SOC2018.1": "SOC_related"}),
    left_on="Related O*NET-SOC Code",
    right_on="SOC_related",
    how="left"
)

# Order related_exposure by "Low", "Medium", "High", for each "O*NET-SOC Code" group
rank_map = {
    "Very Low": 1,
    "Low": 2,
    "Medium": 3,
    "High": 4,
    "Very High": 5
}

related["exposure_rank"] = related["related_exposure"].map(rank_map)

related = related.sort_values(
    by=["Title", "exposure_rank"],
    ascending=[True, True]
)

# Keep unique O*NET-SOC Code and Related O*NET-SOC Code retain all variables
related = related.drop_duplicates(subset=["O*NET-SOC Code", "Related O*NET-SOC Code"])

In [89]:
related

,O*NET-SOC Code,Title,Related O*NET-SOC Code,Related Title,Relatedness Tier,Index,SOC2018.1,exposure,SOC_related,related_exposure,exposure_rank
1992,13-2011,Accountants and Auditors,13-1111,Management Analysts,Supplemental,12,13-2011,Very High,13-1111,Medium,3
1978,13-2011,Accountants and Auditors,13-2061,Financial Examiners,Primary-Short,2,13-2011,Very High,13-2061,High,4
1979,13-2011,Accountants and Auditors,11-3031,Treasurers and Controllers,Primary-Short,3,13-2011,Very High,11-3031,High,4
1986,13-2011,Accountants and Auditors,13-2031,Budget Analysts,Primary-Long,6,13-2011,Very High,13-2031,High,4
1988,13-2011,Accountants and Auditors,13-2052,Personal Financial Advisors,Primary-Long,8,13-2011,Very High,13-2052,High,4
...,...,...,...,...,...,...,...,...,...,...,...
5407,19-1023,Zoologists and Wildlife Biologists,25-1053,"Environmental Science Teachers, Postsecondary",Supplemental,16,19-1023,High,25-1053,High,4
5408,19-1023,Zoologists and Wildlife Biologists,25-1041,"Agricultural Sciences Teachers, Postsecondary",Supplemental,17,19-1023,High,25-1041,High,4
5410,19-1023,Zoologists and Wildlife Biologists,19-2043,Hydrologists,Supplemental,19,19-1023,High,19-2043,High,4
5375,19-1023,Zoologists and Wildlife Biologists,19-1029,Biologists,Primary-Short,1,19-1023,High,19-1029,Very High,5


In [90]:
related = related.sort_values(
    ["O*NET-SOC Code", "related_exposure"],
    ascending=[True, True]
).copy()

# 2) Create a fresh per-group order based on the CURRENT row order
related["rel_num"] = related.groupby("O*NET-SOC Code").cumcount() + 1

# 3) Pivot wider using the fresh sequence
related_pivot = related.pivot(
    index="O*NET-SOC Code",
    columns="rel_num",
    values=["Related Title", "Related O*NET-SOC Code"]
)

# 4) Flatten columns
related_pivot.columns = [
    f"{col1}_{col2}" for col1, col2 in related_pivot.columns
]

# Turn O*NET-SOC Code back into a regular column
related_pivot = related_pivot.reset_index()

# Merge GrowthRate
related_pivot = related_pivot.merge(
    lmi[["SOC Code", "GrowthRate"]],
    left_on="O*NET-SOC Code",
    right_on="SOC Code",
    how="left"
)

# Flag above-average growth
related_pivot["GrowthRateAboveAvg"] = (
    related_pivot["GrowthRate"] > related_pivot["GrowthRate"].mean()
)

In [91]:
related_pivot

,O*NET-SOC Code,Related Title_1,Related Title_2,Related Title_3,Related Title_4,Related Title_5,Related Title_6,Related Title_7,Related Title_8,Related Title_9,...,Related O*NET-SOC Code_14,Related O*NET-SOC Code_15,Related O*NET-SOC Code_16,Related O*NET-SOC Code_17,Related O*NET-SOC Code_18,Related O*NET-SOC Code_19,Related O*NET-SOC Code_20,SOC Code,GrowthRate,GrowthRateAboveAvg
0,11-1011,Management Analysts,Social and Community Service Managers,Administrative Services Managers,General and Operations Managers,Treasurers and Controllers,Compliance Managers,"Education Administrators, Postsecondary",Chief Sustainability Officers,First-Line Supervisors of Office and Administr...,...,43-6011,41-1012,27-3031,13-1075,43-6014,11-1031,NaN,11-1011,1.4890,True
1,11-1021,First-Line Supervisors of Housekeeping and Jan...,Administrative Services Managers,Industrial Production Managers,Facilities Managers,First-Line Supervisors of Entertainment and Re...,"First-Line Supervisors of Mechanics, Installer...",First-Line Supervisors of Production and Opera...,Management Analysts,First-Line Supervisors of Construction Trades ...,...,43-1011,41-1012,13-1082,15-1299,17-2112,53-1042,53-1043,11-1021,0.8106,False
2,11-2011,Management Analysts,Merchandise Displayers and Window Trimmers,Marketing Managers,Market Research Analysts and Marketing Special...,Sales Managers,Fundraisers,Art Directors,Advertising Sales Agents,Public Relations Specialists,...,41-3091,13-1011,11-3061,41-9041,15-1299,15-2051,NaN,11-2011,0.5038,False
3,11-2021,Management Analysts,Market Research Analysts and Marketing Special...,Sales Managers,Advertising and Promotions Managers,Financial and Investment Analysts,General and Operations Managers,Advertising Sales Agents,Public Relations Managers,Public Relations Specialists,...,41-3031,13-1082,13-1081,27-3043,15-2051,13-1022,NaN,11-2021,0.8244,True
4,11-2022,Management Analysts,Marketing Managers,Market Research Analysts and Marketing Special...,Advertising and Promotions Managers,First-Line Supervisors of Retail Sales Workers,General and Operations Managers,Chief Executives,Financial and Investment Analysts,First-Line Supervisors of Office and Administr...,...,13-1199,43-4051,41-3031,13-1081,43-4011,13-1022,13-1023,11-2022,0.4360,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,43-9081,"Office Machine Operators, Except Computer",Archivists,Editors,Desktop Publishers,Technical Writers,Word Processors and Typists,File Clerks,Writers and Authors,Data Entry Keyers,...,27-3092,43-9111,25-4022,19-4061,31-9094,43-4021,NaN,43-9081,-1.7393,False
246,43-9111,Management Analysts,Statisticians,"Bookkeeping, Accounting, and Auditing Clerks",Document Management Specialists,Health Information Technologists and Medical R...,Social Science Research Assistants,Bioinformatics Technicians,Database Architects,File Clerks,...,15-1211,15-2031,15-1242,15-2051,NaN,NaN,NaN,43-9111,-0.3945,False
247,51-6092,"Sewers, Hand","Tailors, Dressmakers, and Custom Sewers","Cutters and Trimmers, Hand",Sewing Machine Operators,"Molders, Shapers, and Casters, Except Metal an...",Shoe and Leather Workers and Repairers,"Painting, Coating, and Decorating Workers",Shoe Machine Operators and Tenders,"Patternmakers, Metal and Plastic",...,27-1012,51-4111,51-4061,51-6062,27-1022,27-1021,NaN,51-6092,0.0000,False
248,51-9162,"Multiple Machine Tool Setters, Operators, and ...",Engine and Other Machine Assemblers,Computer Numerically Controlled Tool Operators,Machinists,"Model Makers, Metal and Plastic","Patternmakers, Metal and Plastic","Milling and Planing Machine Setters, Operators...","Lathe and Turning Machine Tool Setters, Operat...",Robotics Technicians,...,17-2199,17-2061,15-1251,51-2023,51-2022,NaN,NaN,51-9162,1.5191,True


In [92]:
def make_id(title):
    return re.sub(r'[^a-z0-9]+', '-', str(title).lower()).strip('-')

cols = [c for c in related_pivot.columns if c.startswith("Related Title_")]

if not cols:
    raise ValueError("No columns found starting with 'Related Title_'")

related_pivot["related_titles_clean"] = related_pivot[cols].apply(
    lambda row: ", ".join(
        f'"{make_id(val)}"'
        for val in row
        if pd.notna(val) and str(val).strip() != ""
    ),
    axis=1
)

In [93]:
df2 = pd.merge(df, related_pivot[["O*NET-SOC Code", "GrowthRateAboveAvg", "related_titles_clean"]], left_on="SOC2018.1", right_on="O*NET-SOC Code", how="left")


In [94]:
# Clean up names
df2.rename(columns={"Title": "title",
                   'SOC2018.1':"soc", 
                   "Description": "summary",
                   "50th Percentile Wage": "medianWage",
                   "Annual Total Openings": "annualOpenings",
                   "Rounded Employment": "employment", 
                   "Annual Percent Change": "projectedGrowth",
                   "EducationValue": "typicalEducation"
                   }, inplace=True)

In [95]:
df2.head()

,soc,PCA Weighted Score.1,Z Score Variance.1,O*NET-SOC Code_x,title,summary,SOC Code,annualOpenings,projectedGrowth,typicalEducation,WorkExperienceValue,GrowthRate,SOC Code1,medianWage,employment,exposure,O*NET-SOC Code_y,GrowthRateAboveAvg,related_titles_clean
0,11-1011,0.818306,0.395745,11-1011,Chief Executives,Determine and formulate policies and provide o...,11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High,11-1011,True,"""management-analysts"", ""social-and-community-s..."
1,11-1011,0.818306,0.395745,11-1011,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High,11-1011,True,"""management-analysts"", ""social-and-community-s..."
2,11-1021,0.696313,0.213237,11-1021,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",11-1021,"8,878",0.81%,Bachelor's degree,5 years or more,0.8106,11-1021,"$90,000","100,340",High,11-1021,False,"""first-line-supervisors-of-housekeeping-and-ja..."
3,11-2011,0.678133,0.185190,11-2011,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",11-2011,8,0.5%,Bachelor's degree,Less than 5 years,0.5038,11-2011,"$107,000",50,High,11-2011,False,"""management-analysts"", ""merchandise-displayers..."
4,11-2021,0.783175,0.237888,11-2021,Marketing Managers,"Plan, direct, or coordinate marketing policies...",11-2021,468,0.82%,Bachelor's degree,5 years or more,0.8244,11-2021,"$135,920","4,350",High,11-2021,True,"""management-analysts"", ""market-research-analys..."


## Build occupation-level website JSON

This section converts the cleaned occupation dataset into the final JSON structure used by the website. Each occupation is grouped into one record with its core metadata, labor market indicators, related occupation IDs, and any attached training program information.

The output from this step is `data.json`, which powers the occupation detail pages and search/explorer views.

In [96]:
def parse_related(val):
    if pd.isna(val) or str(val).strip() == "":
        return []
    
    return [
        x.strip().strip('"')
        for x in str(val).split(",")
        if x.strip() != ""
    ]

def make_id(title):
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

grouped = df2.groupby([
    "title", "soc", "exposure", "summary",
    "medianWage", "annualOpenings",
    "employment", "projectedGrowth",
    "typicalEducation"
], dropna=False)

output = []

for keys, group in grouped:
    (
        title, soc, exposure, summary,
        medianWage, annualOpenings,
        employment, projectedGrowth,
        typicalEducation
    ) = keys

    training = []
    for _, row in group.iterrows():
        if pd.notna(row.get("provider")):
            training.append({
                "provider": row.get("provider"),
                "program": row.get("program"),
                "cip": row.get("cip"),
                "award": row.get("award"),
                "location": row.get("location")
            })

    related_ids = (
        parse_related(group["related_titles_clean"].dropna().iloc[0])
        if group["related_titles_clean"].notna().any()
        else []
    )

    obj = {
        "id": make_id(title),
        "title": title,
        "soc": soc,
        "exposure": exposure,
        "summary": summary,
        "laborMarket": {
            "medianWage": str(medianWage),
            "annualOpenings": str(annualOpenings),
            "employment": str(employment),
            "projectedGrowth": str(projectedGrowth),
            "typicalEducation": typicalEducation
        },
        "relatedOccupationIds": related_ids,
        "training": training
    }

    output.append(obj)

with open("data.json", "w") as f:
    json.dump(output, f, indent=2)

C:\Users\301533\AppData\Local\Temp\ipykernel_37252\863912508.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df2.groupby([


## Build occupation master dataset

This section creates the core occupation-level master dataset used throughout the AI website and Sankey transition pipeline.

The dataset combines:
- occupation identifiers and titles
- AI exposure categories
- occupation summaries/descriptions
- Arizona labor market metrics
- education and work experience requirements

The resulting dataframe is later merged into the Sankey pathway outputs and website occupation pages to provide labor market context for each transition pathway.

In [97]:
import pandas as pd
import json
from pathlib import Path

# =====================================================
# BUILD OCCUPATION MASTER DATA
# =====================================================

def make_id(title):
    return re.sub(r'[^a-z0-9]+', '-', str(title).lower()).strip('-')

ferris_occ = df[[
    "SOC2018.1",
    "Title",
    "Description",
    "exposure",
    "50th Percentile Wage",
    "Annual Total Openings",
    "Rounded Employment",
    "Annual Percent Change",
    "EducationValue",
    "WorkExperienceValue",
    "GrowthRate"
]].copy()

ferris_occ = ferris_occ.rename(columns={
    "SOC2018.1": "soc",
    "Title": "title",
    "Description": "summary",
    "50th Percentile Wage": "medianWage",
    "Annual Total Openings": "annualOpenings",
    "Rounded Employment": "employment",
    "Annual Percent Change": "projectedGrowth",
    "EducationValue": "typicalEducation",
    "WorkExperienceValue": "workExperience",
    "GrowthRate": "growthRate"
})


ferris_occ["id"] = ferris_occ["title"].apply(make_id)

ferris_occ.head()

,soc,title,summary,exposure,medianWage,annualOpenings,employment,projectedGrowth,typicalEducation,workExperience,growthRate,id
0,11-1011,Chief Executives,Determine and formulate policies and provide o...,High,"$150,590",462,"3,100",1.49%,Bachelor's degree,5 years or more,1.4890,chief-executives
1,11-1011,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",High,"$150,590",462,"3,100",1.49%,Bachelor's degree,5 years or more,1.4890,chief-sustainability-officers
2,11-1021,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",High,"$90,000","8,878","100,340",0.81%,Bachelor's degree,5 years or more,0.8106,general-and-operations-managers
3,11-2011,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",High,"$107,000",8,50,0.5%,Bachelor's degree,Less than 5 years,0.5038,advertising-and-promotions-managers
4,11-2021,Marketing Managers,"Plan, direct, or coordinate marketing policies...",High,"$135,920",468,"4,350",0.82%,Bachelor's degree,5 years or more,0.8244,marketing-managers


## Load finalized related occupation transition pathways

This section loads the finalized related occupation transition dataset developed for the AI transition pathway analysis.

The workbook contains:
- direct lower-exposure occupation transitions
- intermediary/gateway transition pathways
- Lightcast and O*NET related occupation relationships
- compatibility and relationship strength metrics
- AI exposure classifications for source and destination occupations

This dataset serves as the primary input for generating the Sankey transition visualizations and occupation pathway explorer.

In [98]:
# =====================================================
# LOAD ARIELLE SANKEY / RELATED OCCUPATION FILE
# =====================================================

sankey_file = Path(
    r"G:\.shortcut-targets-by-id\1KKrBsNyhwzbYus9ExtUkUd_3jE4VZ4W6\AI_Analysis\Arielle's Stuff\High_VeryHigh_AND_Medium_Related_Occs_Final.xlsx"
)

sankey = pd.read_excel(sankey_file)

sankey.head()

,Source_SOC,Source_Occupation,Source_AI_Exposure_Group,Transition_Type,Gateway_SOC,Gateway_Occupation,Gateway_AI_Exposure_Group,Target_SOC,Target_Occupation,Target_AI_Exposure_Group,Destination_SOC,Destination_Occupation,Destination_AI_Exposure_Group,Final_Pathway_Score_100,Recommendation_Flag,Gateway_Recommendation_Flag,Source_Type,ONET_Tier,Final_Compat_Index
0,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3012,Administrative Services Managers,Medium,11-3012,Administrative Services Managers,Medium,100.0,Recommend,NaN,Both,Primary-Short,95.0
1,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3013,Facilities Managers,Medium,11-3013,Facilities Managers,Medium,52.5,Recommend,NaN,ONET,Primary-Short,NaN
2,11-1021,General and Operations Managers,High,Direct,NaN,NaN,NaN,11-3051,Industrial Production Managers,Medium,11-3051,Industrial Production Managers,Medium,49.1,Recommend,NaN,ONET,Primary-Long,NaN
3,11-2011,Advertising and Promotions Managers,High,Direct,NaN,NaN,NaN,13-1111,Management Analysts,Medium,13-1111,Management Analysts,Medium,77.7,Recommend,NaN,Both,Supplemental,94.0
4,11-2021,Marketing Managers,High,Direct,NaN,NaN,NaN,13-1111,Management Analysts,Medium,13-1111,Management Analysts,Medium,73.1,Recommend,NaN,Both,Supplemental,95.0


In [99]:
sankey.columns.tolist()

['Source_SOC',
 'Source_Occupation',
 'Source_AI_Exposure_Group',
 'Transition_Type',
 'Gateway_SOC',
 'Gateway_Occupation',
 'Gateway_AI_Exposure_Group',
 'Target_SOC',
 'Target_Occupation',
 'Target_AI_Exposure_Group',
 'Destination_SOC',
 'Destination_Occupation',
 'Destination_AI_Exposure_Group',
 'Final_Pathway_Score_100',
 'Recommendation_Flag',
 'Gateway_Recommendation_Flag',
 'Source_Type',
 'ONET_Tier',
 'Final_Compat_Index']

## Merge occupation-level labor market data into transition pathways

This section merges the occupation master dataset into the finalized Sankey transition pathways file.

Labor market and occupational attributes are attached separately for:

- source occupations
- destination occupations
- intermediary/gateway occupations

This enriches each transition pathway with:
- wages
- employment
- annual openings
- projected growth
- education requirements
- work experience requirements
- occupation summaries

The merged dataset becomes the primary analytical file used for Sankey visualization generation and website transition outputs.

In [100]:
# =====================================================
# MERGE FERRIS OCCUPATION DATA INTO SANKEY DATA
# =====================================================

# SOURCE OCCUPATIONS
sankey2 = sankey.merge(
    ferris_occ.add_prefix("source_"),
    left_on="Source SOC",
    right_on="source_soc",
    how="left"
)

# DESTINATION OCCUPATIONS
sankey2 = sankey2.merge(
    ferris_occ.add_prefix("dest_"),
    left_on="Related Occupation SOC",
    right_on="dest_soc",
    how="left"
)

# GATEWAY OCCUPATIONS
sankey2 = sankey2.merge(
    ferris_occ.add_prefix("gateway_"),
    left_on="Gateway SOC",
    right_on="gateway_soc",
    how="left"
)

print(sankey2.shape)

sankey2.head()

KeyError: 'Source SOC'

In [ ]:
sankey2.columns.tolist()

['Pathway Type',
 'Source SOC',
 'Source Occupation',
 'Source AI Exposure',
 'Gateway Rank',
 'Gateway SOC',
 'Gateway Occupation',
 'Gateway AI Exposure',
 'Related Occupation SOC',
 'Related Occupation',
 'Related Occupation AI Exposure',
 'Relationship Source',
 'Lightcast Rank',
 'Related Occupation O*NET Rank',
 'Average Source Rank',
 'Combined Rank',
 'Lightcast Compatibility Index',
 'Related Occupation O*NET Tier',
 'Related Occupation O*NET Index',
 'Lightcast Score',
 'Related Occupation O*NET Score',
 'Combined Lightcast + O*NET Score',
 'Combined Score 0-100',
 'source_soc',
 'source_title',
 'source_summary',
 'source_exposure',
 'source_medianWage',
 'source_annualOpenings',
 'source_employment',
 'source_projectedGrowth',
 'source_typicalEducation',
 'source_workExperience',
 'source_growthRate',
 'source_id',
 'dest_soc',
 'dest_title',
 'dest_summary',
 'dest_exposure',
 'dest_medianWage',
 'dest_annualOpenings',
 'dest_employment',
 'dest_projectedGrowth',
 'dest_ty

In [ ]:
sankey2[
    [
        "Source Occupation",
        "source_medianWage",
        "source_annualOpenings",
        "source_employment",
        "Related Occupation",
        "dest_medianWage",
        "dest_annualOpenings",
        "dest_employment",
        "Gateway Occupation",
        "gateway_medianWage"
    ]
].head(20)

,Source Occupation,source_medianWage,source_annualOpenings,source_employment,Related Occupation,dest_medianWage,dest_annualOpenings,dest_employment,Gateway Occupation,gateway_medianWage
0,Chief Executives,"$150,590",462,"3,100",Social and Community Service Managers,"$75,070",345,"2,870",NaN,NaN
1,Chief Executives,"$150,590",462,"3,100",Social and Community Service Managers,"$75,070",345,"2,870",NaN,NaN
2,Chief Executives,"$150,590",462,"3,100",Legislators,NaN,NaN,NaN,NaN,NaN
3,Chief Executives,"$150,590",462,"3,100",Legislators,NaN,NaN,NaN,NaN,NaN
4,Chief Executives,"$150,590",462,"3,100",Management Analysts,"$89,520","1,896","18,110",NaN,NaN
5,Chief Executives,"$150,590",462,"3,100",Management Analysts,"$89,520","1,896","18,110",NaN,NaN
6,Chief Executives,"$150,590",462,"3,100",Administrative Services Managers,"$99,800",509,"6,310",NaN,NaN
7,Chief Executives,"$150,590",462,"3,100",Administrative Services Managers,"$99,800",509,"6,310",NaN,NaN
8,Chief Executives,"$150,590",462,"3,100",First-Line Supervisors of Entertainment and Re...,"$46,120",429,"2,110",NaN,NaN
9,Chief Executives,"$150,590",462,"3,100",First-Line Supervisors of Entertainment and Re...,"$46,120",429,"2,110",NaN,NaN


In [ ]:
output_path = Path(
    r"G:\.shortcut-targets-by-id\1KKrBsNyhwzbYus9ExtUkUd_3jE4VZ4W6\AI_Analysis\Arielle's Stuff\Sankey_With_Ferris_Economics_TEST.xlsx"
)

sankey2.to_excel(output_path, index=False)

In [ ]:
# pathways_output = Path(
#     r"C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\pathways.json"
# )

# sankey2.to_json(
#     pathways_output,
#     orient="records",
#     indent=2
# )

In [ ]:
# pathways_output = Path(
#     r"C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\pathways.json"
# )

# sankey2.to_json(
#     pathways_output,
#     orient="records",
#     indent=2
# )

# print(f"Saved to: {pathways_output}")

Saved to: C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\pathways.json


In [2]:
# =====================================================
# CREATE pathways.json FROM UPDATED PRODUCTION WORKBOOK
# =====================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np
import re

EXCEL_FILE = Path(
    r"G:\.shortcut-targets-by-id\1KKrBsNyhwzbYus9ExtUkUd_3jE4VZ4W6\AI_Analysis\Arielle's Stuff\High_VeryHigh_AND_Medium_Related_Occs_Final.xlsx"
)

OUT_PATHWAYS = Path(
    r"C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\pathways.json"
)

def make_id(title):
    return re.sub(r"[^a-z0-9]+", "-", str(title).lower()).strip("-")

def clean_money_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace("$", "").replace(",", "").strip()
    if s.lower() in ["", "nan", "none", "n/a", "na", "-"]:
        return np.nan
    return pd.to_numeric(s, errors="coerce")

def fmt_money(x):
    if pd.isna(x):
        return None
    return f"${x:,.0f}"

def fmt_wage_diff(x):
    if pd.isna(x):
        return "N/A"
    sign = "+" if x >= 0 else "-"
    return f"{sign}${abs(x):,.0f}"

def fmt_openings(x):
    if pd.isna(x):
        return None
    return f"{x:,.0f}"

def fmt_growth(x):
    if pd.isna(x):
        return None
    return f"{x:.2f}%"

def clean_json_value(x):
    if pd.isna(x):
        return None
    return x

# =====================================================
# LOAD UPDATED DIRECT + GATEWAY SHEETS
# =====================================================

direct = pd.read_excel(EXCEL_FILE, sheet_name="Recommended_Direct_Only")
gateway = pd.read_excel(EXCEL_FILE, sheet_name="Recommended_Gateway_Top3")

direct.columns = direct.columns.astype(str).str.strip()
gateway.columns = gateway.columns.astype(str).str.strip()

# =====================================================
# BUILD DIRECT WEBSITE ROWS
# =====================================================

direct_rows = direct.copy()

direct_rows["source_id"] = direct_rows["Source_Occupation"].apply(make_id)
direct_rows["dest_id"] = direct_rows["Target_Occupation"].apply(make_id)

direct_rows["source_soc"] = direct_rows["Source_SOC"]
direct_rows["source_title"] = direct_rows["Source_Occupation"]
direct_rows["dest_soc"] = direct_rows["Target_SOC"]
direct_rows["dest_title"] = direct_rows["Target_Occupation"]
direct_rows["dest_exposure"] = direct_rows["Target_AI_Exposure_Group"]
direct_rows["table_pathway"] = "Direct"

direct_rows["score_for_sort"] = pd.to_numeric(
    direct_rows["direct_realism_score_100"],
    errors="coerce"
)

direct_rows["source_wage_num"] = pd.to_numeric(direct_rows["source_wage"], errors="coerce")
direct_rows["dest_wage_num"] = pd.to_numeric(direct_rows["target_wage"], errors="coerce")
direct_rows["wage_diff_num"] = direct_rows["dest_wage_num"] - direct_rows["source_wage_num"]

direct_rows["dest_openings_num"] = pd.to_numeric(
    direct_rows.get("AI_Annual_Total_Openings"),
    errors="coerce"
)

direct_rows["dest_growth_num"] = pd.to_numeric(
    direct_rows.get("AI_Growth_Rate"),
    errors="coerce"
)

direct_rows["median_wage_clean"] = direct_rows["dest_wage_num"].apply(fmt_money)
direct_rows["wage_diff"] = direct_rows["wage_diff_num"].apply(fmt_wage_diff)
direct_rows["openings_clean"] = direct_rows["dest_openings_num"].apply(fmt_openings)
direct_rows["growth_clean"] = direct_rows["dest_growth_num"].apply(fmt_growth)

direct_rows["destination_education"] = direct_rows.get("AI_Education_Value", "N/A")

# =====================================================
# BUILD GATEWAY WEBSITE ROWS
# Destination shown in the ranked table = final destination
# =====================================================

gateway_rows = gateway.copy()

gateway_rows["source_id"] = gateway_rows["Source_Occupation"].apply(make_id)
gateway_rows["dest_id"] = gateway_rows["Destination_Occupation"].apply(make_id)

gateway_rows["source_soc"] = gateway_rows["Source_SOC"]
gateway_rows["source_title"] = gateway_rows["Source_Occupation"]
gateway_rows["dest_soc"] = gateway_rows["Destination_SOC"]
gateway_rows["dest_title"] = gateway_rows["Destination_Occupation"]
gateway_rows["dest_exposure"] = gateway_rows["Destination_AI_Exposure_Group"]
gateway_rows["table_pathway"] = "Stepping-stone pathway"

gateway_rows["score_for_sort"] = pd.to_numeric(
    gateway_rows["gateway_pathway_score_100"],
    errors="coerce"
)

gateway_rows["source_wage_num"] = pd.to_numeric(gateway_rows["source_wage"], errors="coerce")
gateway_rows["dest_wage_num"] = pd.to_numeric(gateway_rows["destination_wage"], errors="coerce")
gateway_rows["wage_diff_num"] = gateway_rows["dest_wage_num"] - gateway_rows["source_wage_num"]

gateway_rows["dest_openings_num"] = pd.to_numeric(
    gateway_rows.get("AI_Annual_Total_Openings"),
    errors="coerce"
)

gateway_rows["dest_growth_num"] = pd.to_numeric(
    gateway_rows.get("AI_Growth_Rate"),
    errors="coerce"
)

gateway_rows["median_wage_clean"] = gateway_rows["dest_wage_num"].apply(fmt_money)
gateway_rows["wage_diff"] = gateway_rows["wage_diff_num"].apply(fmt_wage_diff)
gateway_rows["openings_clean"] = gateway_rows["dest_openings_num"].apply(fmt_openings)
gateway_rows["growth_clean"] = gateway_rows["dest_growth_num"].apply(fmt_growth)

gateway_rows["destination_education"] = gateway_rows.get("AI_Education_Value", "N/A")
gateway_rows["gateway_occupation"] = gateway_rows["Gateway_Occupation"]
gateway_rows["gateway_soc"] = gateway_rows["Gateway_SOC"]

# =====================================================
# COMBINE + RANK
# =====================================================

keep_cols = [
    "source_id",
    "source_soc",
    "source_title",
    "dest_id",
    "dest_soc",
    "dest_title",
    "dest_exposure",
    "table_pathway",
    "score_for_sort",
    "median_wage_clean",
    "wage_diff",
    "openings_clean",
    "growth_clean",
    "destination_education",
    "gateway_occupation",
    "gateway_soc"
]

for c in keep_cols:
    if c not in direct_rows.columns:
        direct_rows[c] = None
    if c not in gateway_rows.columns:
        gateway_rows[c] = None

df_paths = pd.concat(
    [direct_rows[keep_cols], gateway_rows[keep_cols]],
    ignore_index=True
)

df_paths = df_paths[df_paths["dest_id"].notna()].copy()

df_paths = df_paths.sort_values(
    by=[
        "source_id",
        "score_for_sort",
        "table_pathway"
    ],
    ascending=[True, False, True],
    na_position="last"
)

df_paths = df_paths.drop_duplicates(
    subset=["source_id", "dest_id", "table_pathway"],
    keep="first"
)
# =====================================================
# FALLBACK: ADD TOP 3 WEAK DIRECT MATCHES FOR SOURCES
# WITH NO RECOMMENDED DIRECT OR GATEWAY OPTIONS
# =====================================================

all_direct = pd.read_excel(EXCEL_FILE, sheet_name="Final_Transitions")
all_direct.columns = all_direct.columns.astype(str).str.strip()

# Sources already covered by recommended direct/gateway rows
covered_sources = set(df_paths["source_soc"].astype(str).str.strip())

# Build fallback rows from weak/review direct rows
fallback = all_direct[
    ~all_direct["Source_SOC"].astype(str).str.strip().isin(covered_sources)
].copy()

fallback = fallback[
    fallback["Target_AI_Exposure_Group"].isin(["Medium", "Low", "Very Low"])
].copy()

fallback["source_id"] = fallback["Source_Occupation"].apply(make_id)
fallback["dest_id"] = fallback["Target_Occupation"].apply(make_id)

fallback["source_soc"] = fallback["Source_SOC"]
fallback["source_title"] = fallback["Source_Occupation"]
fallback["dest_soc"] = fallback["Target_SOC"]
fallback["dest_title"] = fallback["Target_Occupation"]
fallback["dest_exposure"] = fallback["Target_AI_Exposure_Group"]
fallback["table_pathway"] = "Additional related option"

fallback["score_for_sort"] = pd.to_numeric(
    fallback["direct_realism_score_100"],
    errors="coerce"
)

fallback["source_wage_num"] = pd.to_numeric(fallback["source_wage"], errors="coerce")
fallback["dest_wage_num"] = pd.to_numeric(fallback["target_wage"], errors="coerce")
fallback["wage_diff_num"] = fallback["dest_wage_num"] - fallback["source_wage_num"]

fallback["median_wage_clean"] = fallback["dest_wage_num"].apply(fmt_money)
fallback["wage_diff"] = fallback["wage_diff_num"].apply(fmt_wage_diff)

fallback["dest_openings_num"] = pd.to_numeric(
    fallback.get("AI_Annual_Total_Openings"),
    errors="coerce"
)

fallback["dest_growth_num"] = pd.to_numeric(
    fallback.get("AI_Growth_Rate"),
    errors="coerce"
)

fallback["openings_clean"] = fallback["dest_openings_num"].apply(fmt_openings)
fallback["growth_clean"] = fallback["dest_growth_num"].apply(fmt_growth)

fallback["destination_education"] = fallback.get("AI_Education_Value", "N/A")
fallback["gateway_occupation"] = None
fallback["gateway_soc"] = None

# Keep top 3 weak matches per source
fallback = (
    fallback
    .sort_values(["source_soc", "score_for_sort"], ascending=[True, False])
    .groupby("source_soc")
    .head(3)
    .copy()
)

for c in keep_cols:
    if c not in fallback.columns:
        fallback[c] = None

df_paths = pd.concat(
    [df_paths, fallback[keep_cols]],
    ignore_index=True
)

df_paths = df_paths[df_paths["dest_id"].notna()].copy()

print("Fallback weak-match rows added:", len(fallback))
df_paths["rank"] = df_paths.groupby("source_id").cumcount() + 1
df_paths = df_paths[df_paths["rank"] <= 10].copy()

# =====================================================
# FINAL JSON
# =====================================================

pathways_json = []

for _, r in df_paths.iterrows():
    pathway_label = r.get("table_pathway")

    if pathway_label == "Intermediary" and pd.notna(r.get("gateway_occupation")):
        pathway_label = f"Intermediary via {r.get('gateway_occupation')}"

    pathways_json.append({
        "source_id": r.get("source_id"),
        "source_soc": r.get("source_soc"),
        "source_title": r.get("source_title"),
        "rank": int(r.get("rank")),
        "occupation": r.get("dest_title"),
        "soc": r.get("dest_soc"),
        "pathway": pathway_label,
        "ai_exposure": r.get("dest_exposure"),
        "median_wage": r.get("median_wage_clean"),
        "wage_diff": r.get("wage_diff"),
        "openings": r.get("openings_clean"),
        "growth": r.get("growth_clean"),
        "destination_education": r.get("destination_education"),
        "gateway_occupation": r.get("gateway_occupation"),
        "gateway_soc": r.get("gateway_soc"),
        "link": f"occupation.html?id={r.get('dest_id')}"
    })

pathways_json = [
    {k: clean_json_value(v) for k, v in row.items()}
    for row in pathways_json
]

OUT_PATHWAYS.parent.mkdir(parents=True, exist_ok=True)

OUT_PATHWAYS.write_text(
    json.dumps(pathways_json, indent=2, allow_nan=False),
    encoding="utf-8"
)

print(f"Saved {len(pathways_json):,} ranked transition rows to:")
print(OUT_PATHWAYS)

all_sources = all_direct[
    all_direct["Source_AI_Exposure_Group"].isin(["High", "Very High"])
][["Source_SOC", "Source_Occupation"]].drop_duplicates()

path_sources = set(df_paths["source_soc"].astype(str).str.strip())

missing_from_website = all_sources[
    ~all_sources["Source_SOC"].astype(str).str.strip().isin(path_sources)
]

print("High/Very High sources missing from website table:", len(missing_from_website))
display(missing_from_website)

Fallback weak-match rows added: 342
Saved 1,383 ranked transition rows to:
C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\pathways.json
High/Very High sources missing from website table: 0


,Source_SOC,Source_Occupation


In [ ]:
check = pd.read_json(
    r"C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\pathways.json"
)

print(check.shape)
print(check[check["source_title"].eq("Accountants and Auditors")][
    ["rank", "occupation", "pathway", "ai_exposure", "median_wage", "wage_diff"]
])

(1041, 16)
   rank           occupation pathway ai_exposure median_wage wage_diff
0     1  Management Analysts  Direct      Medium     $89,520  +$10,900


In [ ]:
print("All direct:")
print(all_direct[all_direct["Source_SOC"].astype(str).str.strip().eq("41-3011")][[
    "Source_SOC", "Source_Occupation", "Target_SOC", "Target_Occupation",
    "direct_realism_score_100", "Recommendation_Flag"
]].sort_values("direct_realism_score_100", ascending=False).head(20))

print("All gateway:")
print(all_gateway[all_gateway["Source_SOC"].astype(str).str.strip().eq("41-3011")][[
    "Source_SOC", "Source_Occupation", "Gateway_SOC", "Gateway_Occupation",
    "Destination_SOC", "Destination_Occupation",
    "gateway_pathway_score_100", "Gateway_Recommendation_Flag"
]].sort_values("gateway_pathway_score_100", ascending=False).head(20))

All direct:
     Source_SOC         Source_Occupation Target_SOC  \
1990    41-3011  Advertising Sales Agents    41-9011   
1991    41-3011  Advertising Sales Agents    41-2031   
1992    41-3011  Advertising Sales Agents    41-9091   
1993    41-3011  Advertising Sales Agents    27-1026   

                                      Target_Occupation  \
1990                Demonstrators and Product Promoters   
1991                                Retail Salespersons   
1992  Door-to-Door Sales Workers, News and Street Ve...   
1993         Merchandise Displayers and Window Trimmers   

      direct_realism_score_100  Recommendation_Flag  
1990                      34.2  Review / Weak Match  
1991                      33.0  Review / Weak Match  
1992                      30.7  Review / Weak Match  
1993                      25.2  Review / Weak Match  
All gateway:
Empty DataFrame
Columns: [Source_SOC, Source_Occupation, Gateway_SOC, Gateway_Occupation, Destination_SOC, Destination_Occupatio

In [101]:
check = pd.read_json(
    r"C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\pathways.json"
)

print(check[check["source_soc"].astype(str).eq("41-3011")][
    ["rank", "occupation", "pathway", "ai_exposure", "median_wage", "wage_diff"]
])

      rank                                         occupation  \
1340     1                Demonstrators and Product Promoters   
1341     2                                Retail Salespersons   
1342     3  Door-to-Door Sales Workers, News and Street Ve...   

                        pathway ai_exposure median_wage wage_diff  
1340  Additional related option         Low     $33,280  -$21,480  
1341  Additional related option      Medium     $35,320  -$19,440  
1342  Additional related option      Medium        None       N/A  
